# NLP embeddings and topic labels (Role C · Sentence-BERT)

## The idea in one paragraph

A computer cannot read a review, so every review has to become numbers. B's clusters use **TF-IDF**, which counts *which words* appear in a review and how rare each word is. That only sees exact words: *"I gained a lot of weight"* and *"I put on pounds"* share no important word, so TF-IDF treats them as unrelated. **Sentence-BERT** turns every review into a list of **768 numbers (a vector) that captures its meaning**, so reviews that say the same thing in different words end up with similar vectors. We use `FremyCompany/BioLORD-2023`, a Sentence-BERT model trained on biomedical text. This is the **new technique that was not taught in the course** (the course covers TF-IDF, LSH, k-means, DBSCAN and Davies-Bouldin, but not text embeddings).

## What the embeddings are used for

1. **Do clusters built from meaning look like B's clusters built from words?** We run k-means on the embeddings and compare the result with B's 47 clusters (B used TF-IDF, a course topic). The comparison uses the Davies-Bouldin index (how well separated the clusters are), **drug purity** (does a cluster contain almost only one drug? then it groups by drug, not by experience), the **ARI** (how much the two clusterings agree) and the spread of ratings inside a cluster. We repeat it after hiding the drug names to see how much of the grouping was just drug names.
2. **What is each of B's 47 clusters about?** A cluster is only a number, so we describe it with keywords. First, **class-based TF-IDF** (written by hand, the idea behind BERTopic) finds the words that distinguish one cluster from the others. Then a **KeyBERT-style step** uses the embeddings to pick the five of those words that best represent the cluster's meaning, without repeating the same idea. The keywords are evaluated with automatic metrics (NPMI coherence against a random baseline, and embedding cohesion). There is no manual labelling.
3. **Vectors for A.** One vector per `review_id`. A uses them to repeat the *similar-story search* by meaning (cosine LSH, implemented by A) and compare it with the word-based version.

## Why it matters for the project

All three feed research question 1: *can patient reviews be grouped into common experience types?* They show whether grouping by meaning finds experience types or just drug names, and they give A and D their inputs.


## Pipeline overview

| Step | Section | What it does |
|---|---|---|
| Load & scope | C1 | Load A's clean data and B's cluster assignments, keep the four conditions |
| Embeddings | C2 | Sentence-BERT vectors per condition (optionally also for all other reviews) |
| Sanity check | C3 | Rows, IDs, NaN and vector length agree |
| Topic terms | C4 | Class-based TF-IDF from scratch: the most distinctive terms of every B cluster |
| Topic keywords | C5 | KeyBERT-style: the c-TF-IDF candidates ranked by similarity to the cluster centroid, with diversity (MMR) |
| Cluster table | C6 | Reviews, mean rating and top drugs of each B cluster |
| Automatic validation | C7 | NPMI coherence vs random baseline; embedding cohesion |
| DBSCAN | C8 | Why only k-means clusters are labelled |
| Clustering in meaning space | C9 | k-means on embeddings vs B's TF-IDF clusters (DB, drug purity, ARI), with and without drug names; model comparison |
| Save outputs | C10 | Embeddings for A, labels for D, statistics for the report |

### Outputs

| File | Used by |
|---|---|
| `outputs/embeddings_all.npy` + `embeddings_all_ids.parquet` | **A**: one vector per `review_id` |
| `outputs/cluster_labels_final.csv` | **D** and report: one label per k-means cluster |
| `outputs/cluster_topics_ctfidf.csv`, `cluster_labels_sbert_auto.csv`, `cluster_labels_B_full.csv` | Labelling |
| `outputs/sbert_vs_tfidf_summary.csv`, `sbert_masked_v2_summary.csv`, `model_comparison_sample.csv` | Report: evaluation tables |
| `outputs/role_c_run_stats.json` | Report: all numbers |

## 0 · Setup and parameters

All settings live here. Change a value, re-run from top, and every downstream cell picks it up automatically.

In [1]:
import json
import os
import re
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_rows', 120)
pd.set_option('display.width', 200)
NOTEBOOK_STARTED = time.perf_counter()

# where the data is read from and written to  (same layout as Roles A, B and E)
INPUT_DIRS = [Path('/kaggle/input'), Path('data'), Path('../data'), Path('../clustering/outputs')]
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

run_stats = {}   # every reported number lands here; saved as JSON at the end

# ── Scope ─────────────────────────────────────────────────────────────────────
CONDITIONS = ['depression', 'birth control', 'pain', 'acne']   # B's four conditions (lower case)
TEXT, ID, COND = 'review_text_clean', 'review_id', 'condition_name'
CLUSTER_COL = 'kmeans_cluster'         # DBSCAN is not used (see C7)

# ── Sentence-BERT ─────────────────────────────────────────────────────────────
MODEL_NAME = 'FremyCompany/BioLORD-2023'   # biomedical Sentence-BERT (768-d). To go back: 'all-MiniLM-L6-v2' (384-d)
MODEL_SLUG = 'biolord'                      # goes into every file name, so two models never share a cache
EMB        = f'emb_{MODEL_SLUG}'            # embeddings per condition
EMB_MASKED = f'emb_{MODEL_SLUG}_masked_v2'  # same, drug names masked
EMB_OTHER  = f'emb_{MODEL_SLUG}_other'      # other conditions
BATCH_SIZE = 64
EMBED_ALL_OTHER_REVIEWS = False  # also embed the ~92k reviews of other conditions (A's story search uses all 128k). Slow with BioLORD (hours on CPU); True if you have a GPU or time

# ── Labelling ─────────────────────────────────────────────────────────────────
N_CENTRAL, N_RANDOM = 5, 5       # central and random reviews shown per cluster

key = lambda c: c.replace(' ', '_')

## C1 · Load and scope

I load A's clean dataset and create `cond_l` (the condition in lower case, without extra spaces) so that filtering by condition does not fail because of capitalisation.
I only work with 4 conditions (Depression, Birth Control, Pain, Acne) because those are the ones B grouped into clusters.

In [2]:
def find_input(file_name):
    for folder in INPUT_DIRS:
        if folder.exists():
            matches = sorted(folder.rglob(file_name))
            if matches:
                return matches[0]
    raise FileNotFoundError(f'{file_name} not found in {INPUT_DIRS}')

reviews  = pd.read_parquet(find_input('reviews_clean_deduplicated.parquet'))   # Role A
clusters = pd.read_parquet(find_input('cluster_assignments.parquet'))          # Role B
reviews['cond_l'] = reviews[COND].str.lower().str.strip()
run_stats['n_reviews_all'] = int(len(reviews))
run_stats['n_reviews_in_scope'] = int(reviews['cond_l'].isin(CONDITIONS).sum())

print(reviews.shape)
print(reviews.columns.tolist())
print()
print("Reseñas por condición (las 4 mías):")
print(reviews[reviews["cond_l"].isin(CONDITIONS)]["cond_l"].value_counts())
print()
print("Clusters k-means de B por condición:")
print(clusters.groupby(COND)[CLUSTER_COL].nunique())

(128663, 11)
['review_id', 'drug_name', 'drug_name_aliases', 'condition_name', 'rating', 'review_date', 'useful_count', 'review_text_clean', 'review_text_normalised', 'word_count', 'cond_l']

Reseñas por condición (las 4 mías):
cond_l
birth control    19520
depression        7109
pain              5187
acne              4549
Name: count, dtype: int64

Clusters k-means de B por condición:
condition_name
Acne             25
Birth Control    23
Depression       26
Pain             26
Name: kmeans_cluster, dtype: int64


In [3]:
import glob
E_all   = np.load(glob.glob('/kaggle/input/**/embeddings_all.npy', recursive=True)[0])
ids_all = pd.read_parquet(glob.glob('/kaggle/input/**/embeddings_all_ids.parquet', recursive=True)[0])
E_m     = np.load(glob.glob('/kaggle/input/**/embeddings_masked_v2_all.npy', recursive=True)[0])
ids_m   = pd.read_parquet(glob.glob('/kaggle/input/**/embeddings_masked_v2_all_ids.parquet', recursive=True)[0])
pos_all = {r: i for i, r in enumerate(ids_all[ID])}
pos_m   = {r: i for i, r in enumerate(ids_m[ID])}
for c in CONDITIONS:
    ids_c = reviews.loc[reviews["cond_l"] == c, [ID]].reset_index(drop=True)
    np.save(f"{OUTPUT_DIR}/{EMB}_{key(c)}.npy", E_all[[pos_all[r] for r in ids_c[ID]]])
    ids_c.to_parquet(f"{OUTPUT_DIR}/{EMB}_{key(c)}_ids.parquet", index=False)
    np.save(f"{OUTPUT_DIR}/{EMB_MASKED}_{key(c)}.npy", E_m[[pos_m[r] for r in ids_c[ID]]])
    ids_c.to_parquet(f"{OUTPUT_DIR}/{EMB_MASKED}_{key(c)}_ids.parquet", index=False)
print("caché reconstruida")

caché reconstruida


## C2 · Sentence-BERT embeddings

**What it does:** `BioLORD-2023` (a Sentence-BERT model trained on medical concepts) reads a review and returns 768 numbers that represent its *meaning*. Reviews that say the same thing in different words ("I couldn't sleep" / "insomnia") end up close to each other; with TF-IDF they would not.

**Decisions:**
- **Model:** chosen because it is biomedical (it understands clinical terms such as *insomnia*, *libido* and *withdrawal*). It is larger and slower than `all-MiniLM-L6-v2`, which I used in a first version. The small comparison in C9b did not show a clear advantage, so the choice rests on domain fit. Limit: `model.max_seq_length` (printed below); longer reviews are truncated.
- **`normalize_embeddings=True`:** every vector has length 1, so the dot product of two vectors equals their cosine similarity.
- **Input text:** `review_text_clean`, with complete sentences, because the model needs the grammar and not the normalised text.
- **Two files are saved:** a `.npy` file (the numbers) **and** `_ids.parquet` (which `review_id` each row belongs to). Without the IDs file, the `.npy` is useless.
- **Cache:** if the file already exists, it is skipped, so the cell can be re-run without repeating hours of computation. The file name contains the model name.

In [4]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(MODEL_NAME)
print(MODEL_NAME, '| dim:', model.get_sentence_embedding_dimension(), '| max_seq_length:', model.max_seq_length)

for c in CONDITIONS:
    fname = f"{OUTPUT_DIR}/{EMB}_{key(c)}.npy"
    if os.path.exists(fname):
        print(c, "ya hecho, salto"); continue
    sub = reviews[reviews["cond_l"] == c]
    t = time.time()
    emb = model.encode(sub[TEXT].tolist(), batch_size=BATCH_SIZE,
                       show_progress_bar=True, normalize_embeddings=True)
    np.save(fname, emb)
    sub[[ID]].to_parquet(fname.replace(".npy", "_ids.parquet"))
    print(c, emb.shape, f"{time.time()-t:.0f}s")

# El resto de reseñas (otras condiciones): A busca historias parecidas en las 128 663 reseñas, no solo en las mías
fname = f"{OUTPUT_DIR}/{EMB_OTHER}.npy"
if EMBED_ALL_OTHER_REVIEWS and not os.path.exists(fname):
    sub = reviews[~reviews["cond_l"].isin(CONDITIONS)]
    t = time.time()
    emb = model.encode(sub[TEXT].tolist(), batch_size=BATCH_SIZE, show_progress_bar=True, normalize_embeddings=True)
    np.save(fname, emb)
    sub[[ID]].to_parquet(fname.replace(".npy", "_ids.parquet"))
    print("other conditions", emb.shape, f"{time.time()-t:.0f}s")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/7.17k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FremyCompany/BioLORD-2023 | dim: 768 | max_seq_length: 128
depression ya hecho, salto
birth control ya hecho, salto
pain ya hecho, salto
acne ya hecho, salto


## C3 · Sanity check of the embeddings

For each condition I check: (1) the number of rows in the `.npy` file equals the number of IDs and the number of reviews of that condition, (2) there are no NaN values, (3) every vector has norm 1.
If something fails here, everything that follows (clusters, labels, A's LSH) would be misaligned.

In [5]:
check = []
for c in CONDITIONS:
    E   = np.load(f"{OUTPUT_DIR}/{EMB}_{key(c)}.npy")
    ids = pd.read_parquet(f"{OUTPUT_DIR}/{EMB}_{key(c)}_ids.parquet")
    n   = int((reviews["cond_l"] == c).sum())
    check.append({"condition": c, "rows_npy": E.shape[0], "dim": E.shape[1], "n_ids": len(ids), "n_reviews": n,
                  "has_nan": bool(np.isnan(E).any()),
                  "norm_ok": bool(np.allclose(np.linalg.norm(E, axis=1), 1, atol=1e-3)),
                  "ids_unique": bool(ids[ID].is_unique)})
check = pd.DataFrame(check); print(check.to_string(index=False))
assert (check.rows_npy == check.n_ids).all() and (check.n_ids == check.n_reviews).all(), "desalineado"
assert not check.has_nan.any() and check.norm_ok.all() and check.ids_unique.all()
print("OK")

    condition  rows_npy  dim  n_ids  n_reviews  has_nan  norm_ok  ids_unique
   depression      7109  768   7109       7109    False     True        True
birth control     19520  768  19520      19520    False     True        True
         pain      5187  768   5187       5187    False     True        True
         acne      4549  768   4549       4549    False     True        True
OK


## C4 · Topic terms for each cluster: class-based TF-IDF, written by hand

**Context:** B grouped the reviews of each condition into k-means clusters (47 in total). A cluster is only a number, so B described each one in section B7 of the clustering notebook: B read the top-10 TF-IDF terms of the cluster and wrote a label by hand (for example "weight gain" or "Mirena IUD long-term"). Those labels are written in the text of that notebook, and `cluster_profiles.csv` only contains the terms.

**What this section does:** it describes every cluster **automatically and reproducibly**, with a different method, so that it can be compared with B's reading. For each cluster it finds the words (and pairs of consecutive words) that **distinguish it from the other clusters of the same condition**. The method is **class-based TF-IDF (c-TF-IDF)**, the idea behind BERTopic.

**How it works:**
1. All the reviews of a cluster are joined into **one giant document**.
2. We count how many times each word or word pair (such as "weight gain") appears in that document.
3. Each term gets a score:

`score(c, t) = tf(c, t) · log(1 + A / f(t))`

- `tf(c, t)`: the share of the words of cluster `c` that are the term `t` (how common it is **in this cluster**).
- `f(t)`: how many times `t` appears in **all** clusters of the condition together.
- `A`: the average number of words per cluster (it only sets the scale).
- `log(1 + A / f(t))`: a **rarity factor**; it is small if the term appears everywhere and large if it appears almost only in this cluster.

A term therefore scores high if it is **frequent in this cluster and rare in the others**. For example, "effects" appears in every cluster, so it scores low; "pimples" appears mostly in one cluster, so it scores high.

For each cluster we keep the 5, 10 and 30 best terms. The 30 are the candidates from which C5 chooses the final keywords using the embeddings.

**How it differs from B's TF-IDF:** B averages the TF-IDF weight of each word over the reviews of the cluster, which favours frequent words. Here the score compares **cluster against cluster**. The two methods should agree on the essentials, and we measure how much (`word_overlap_with_B`, the overlap between our terms and B's top-10 terms).

**Finding (for the report, not corrected by hand):** in a pilot review of several clusters, the keywords included generic words (*doctor, medication, went, did*), the condition name (*depression*) and drug names (*prozac*). B's own terms show similar artefacts (for example "ve", from "I've", in most Birth Control clusters), so this comes from the text and not only from the method. We do not fix it with a hand-written exclusion list, because there would always be more words and the criterion would be arbitrary. Instead we **measure from the data** the share of keywords that are drug or condition names (`share_name_terms_top10`) and report it as a limitation. This is expected: when a cluster groups the reviews of one drug or one condition, the name is what distinguishes it most.

**Own implementation:** the formula is written with numpy (it is not a function from a package); only the word counting uses `CountVectorizer`.


In [6]:
def c_tf_idf(counts):
    """Class-based TF-IDF (the idea of BERTopic), written by hand.
    counts[c, t] = how many times term t appears in ALL reviews of cluster c (a cluster = one big document).
    score[c, t] = tf[c, t] * log(1 + A / f[t])
      tf[c, t] = counts[c, t] / total words of cluster c     (share of the cluster's words that is t)
      f[t]     = counts of t in all clusters together
      A        = mean number of words per cluster
    A term scores high when it is frequent in this cluster and rare in the other clusters."""
    counts = np.asarray(counts, dtype=float)
    tf = counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
    A = counts.sum(axis=1).mean()
    f = np.maximum(counts.sum(axis=0), 1)
    return tf * np.log(1 + A / f)

In [7]:
import re
from sklearn.feature_extraction.text import CountVectorizer

TEXT_TOPIC = "review_text_normalised" if "review_text_normalised" in reviews.columns else TEXT

# --- nombres de condición y de fármaco, tomados de los datos: NO se excluyen, solo se usan para MEDIR cuántas palabras clave son nombres ---
NAME_WORDS = set()
for col in ["condition_name", "drug_name"]:
    if col in reviews.columns:
        for v in reviews.loc[reviews["cond_l"].isin(CONDITIONS), col].dropna().astype(str).str.lower().unique():
            NAME_WORDS.update(re.findall(r"[a-z]{4,}", v))
NAME_WORDS.update(w for c_ in CONDITIONS for w in re.findall(r"[a-z]{3,}", c_))
prof = pd.read_csv(find_input("cluster_profiles.csv"))                 # términos TF-IDF de B (B7)
prof["condition"] = prof["condition"].str.lower().str.strip()

topic_rows = []
for c in CONDITIONS:
    df = (reviews[reviews["cond_l"] == c][[ID, TEXT_TOPIC]].merge(clusters[[ID, CLUSTER_COL]], on=ID)
          .reset_index(drop=True))
    cv = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=5, max_df=0.5)
    X = cv.fit_transform(df[TEXT_TOPIC].fillna(""))
    cl_ids = np.sort(df[CLUSTER_COL].unique()); pos = {k: i for i, k in enumerate(cl_ids)}
    M = sp.csr_matrix((np.ones(len(df)), (df[CLUSTER_COL].map(pos).values, np.arange(len(df)))),
                      shape=(len(cl_ids), len(df)))
    W = c_tf_idf((M @ X).toarray())                                    # (n_clusters, n_terms)
    terms = np.array(cv.get_feature_names_out())
    for i, cl in enumerate(cl_ids):
        order_terms = np.argsort(W[i])[::-1]
        top = terms[order_terms[:10]]
        top30 = terms[order_terms[:30]]
        b_row = prof[(prof["condition"] == c) & (prof["cluster"] == cl)]
        b_words = set(w for t in b_row["top_10_terms"].iloc[0].split(" | ") for w in t.split()) if len(b_row) else set()
        my_words = set(w for t in top for w in t.split())
        topic_rows.append({"condition": c, "cluster": cl, "n_reviews": int((df[CLUSTER_COL] == cl).sum()),
                           "ctfidf_top5": " | ".join(top[:5]), "ctfidf_top10": " | ".join(top), "ctfidf_top30": " | ".join(top30),
                           "b_top10": " | ".join(t for t in (b_row["top_10_terms"].iloc[0].split(" | ") if len(b_row) else [])),
                           "word_overlap_with_B": round(len(my_words & b_words) / max(len(my_words | b_words), 1), 3)})
topics = pd.DataFrame(topic_rows)
# hallazgo medido: qué parte de las 10 palabras clave son nombres de fármaco/condición (de los datos, sin lista escrita a mano)
topics["share_name_terms_top10"] = topics["ctfidf_top10"].apply(
    lambda s_: round(float(np.mean([any(w in NAME_WORDS for w in t.split()) for t in s_.split(" | ")])), 2))
run_stats["share_top10_terms_that_are_drug_or_condition_names"] = round(float(topics["share_name_terms_top10"].mean()), 3)
run_stats["clusters_with_name_in_top5"] = int(topics["ctfidf_top5"].apply(lambda s_: any(w in NAME_WORDS for t in s_.split(" | ") for w in t.split())).sum())
print("Palabras clave que son nombre de fármaco o condición (media):", run_stats["share_top10_terms_that_are_drug_or_condition_names"],
      "| clusters con alguna en el top 5:", run_stats["clusters_with_name_in_top5"], "de", len(topics))
topics.to_csv(f"{OUTPUT_DIR}/cluster_topics_ctfidf.csv", index=False)
run_stats["ctfidf_word_overlap_with_B_mean"] = round(float(topics["word_overlap_with_B"].mean()), 3)
print("Solapamiento medio de palabras con los términos de B:", run_stats["ctfidf_word_overlap_with_B_mean"])
print(topics[["condition", "cluster", "n_reviews", "ctfidf_top5", "word_overlap_with_B"]].to_string(index=False))

Palabras clave que son nombre de fármaco o condición (media): 0.143 | clusters con alguna en el top 5: 51 de 100
Solapamiento medio de palabras con los términos de B: 0.677
    condition  cluster  n_reviews                                                              ctfidf_top5  word_overlap_with_B
   depression        0        230                   cymbalta | depression | pain | taking cymbalta | years                0.800
   depression        1        249                       pristiq | taking pristiq | feel | taking | effects                0.800
   depression        2        208                      effexor | xr | effexor xr | depression | withdrawal                0.778
   depression        3        272                   wellbutrin | xl | wellbutrin xl | effects | depression                0.889
   depression        4        184                          mood | swings | mood swings | depression | feel                0.636
   depression        5        227           thoughts | suic

## C5 · Keywords for each cluster with Sentence-BERT (KeyBERT-style)

**What it does:** C4 gave us, for each cluster, 30 candidate terms ranked by how distinctive they are. This step chooses the **5 terms that best represent the meaning of the cluster and are specific to it**, using the embeddings. No word list is written by hand: the candidates come from the reviews themselves.

**How it works:**
1. **Candidates:** the 30 best terms (words or word pairs) of the cluster according to the c-TF-IDF of C4.
2. **Vector of each candidate:** every candidate is turned into 768 numbers with the same model (BioLORD) used for the reviews, so terms and reviews live in the same "meaning space".
3. **Two centres:** the **centre of the cluster** (the average of the vectors of its reviews) represents what the cluster is about; the **centre of the condition** (the average of all the reviews of the condition) represents what all its clusters have in common.
4. **Specificity:** for each candidate we compute its cosine similarity to the centre of the cluster **minus** its similarity to the centre of the condition. A term that describes the whole condition (such as "depression" in a depression cluster) is equally close to both centres and scores about 0; a term that is specific to the cluster is closer to its centre and scores high.
5. **Diversity (MMR):** if we only took the highest scores, we would get near-repeats ("weight", "weight gain", "gained weight"). So the five terms are chosen one at a time:
   - The **first** is the one with the highest specificity score.
   - Each next term balances two things: having a high specificity score and **not being similar to the terms already chosen**:
     `score = (1 − diversity) · specificity − diversity · highest similarity to a chosen term`
   - We use `diversity = 0.5`, an equal weight for both.

**What it adds compared with c-TF-IDF alone:** c-TF-IDF ranks terms by **frequency**. This step re-ranks them by **meaning**, using the embeddings; for example, it can prefer "insomnia" over "hours" because it is closer to what the cluster is about. It is the idea of KeyBERT (Grootendorst, 2020), and it is where Sentence-BERT is essential for describing clusters.

**What the specificity step changed:** in a first version without it, the condition name appeared in almost every cluster (for example "depression" in 9 of 12 depression clusters). With it, the condition name disappeared and more specific terms appeared (for example "brain zaps", "degenerative disc"), but more filler words also got through (for example "time", "years", "really").

**How to read the result:** the first keyword is usually the most specific one, and the first two or three are the most useful. The other terms are **different aspects** of the cluster; the order is the order of selection, not a strict ranking. The keywords are saved in `cluster_labels_sbert_auto.csv` (`sbert_keywords`), and the first three form `label_final` in the table for D.

**Limits (important for the report):**
- The candidates can only be terms that already appear in the c-TF-IDF of C4, so drug names and generic words can slip in (see the finding in C4).
- The result is a **suggestion of keywords, not a final label written by a person**, and its last positions often contain filler words.
- The number of candidates (30) and the diversity (0.5) are design choices, not tuned on data.
- The automatic metrics of C7 evaluate the c-TF-IDF terms and the clusters, **not these keywords**; their quality is described qualitatively.



In [8]:
# def keybert_rank(centroid, cand_vecs, cands, top_n=5, diversity=0.5):
#     """Pick top_n candidate terms close to the cluster centroid and different from each other (MMR).
#     score(i) = (1 - diversity) * cos(candidate_i, centroid) - diversity * max cos(candidate_i, already chosen)"""
#     sims = cand_vecs @ centroid
#     chosen = [int(np.argmax(sims))]
#     while len(chosen) < min(top_n, len(cands)):
#         rest = [i for i in range(len(cands)) if i not in chosen]
#         redundancy = (cand_vecs[rest] @ cand_vecs[chosen].T).max(axis=1)
#         score = (1 - diversity) * sims[rest] - diversity * redundancy
#         chosen.append(rest[int(np.argmax(score))])
#     return [cands[i] for i in chosen]


# # embeddings de todos los candidatos (una sola vez)
# all_cands = sorted({t for s_ in topics["ctfidf_top30"] for t in s_.split(" | ")})
# cand_emb = dict(zip(all_cands, model.encode(all_cands, batch_size=BATCH_SIZE, normalize_embeddings=True)))

# rows = []
# for c in CONDITIONS:
#     E   = np.load(f"{OUTPUT_DIR}/{EMB}_{key(c)}.npy")
#     ids = pd.read_parquet(f"{OUTPUT_DIR}/{EMB}_{key(c)}_ids.parquet").reset_index(drop=True)
#     ids["row"] = range(len(ids))
#     df = (ids.merge(clusters[[ID, CLUSTER_COL]], on=ID)
#              .sort_values("row").reset_index(drop=True))
#     for cl, idx in df.groupby(CLUSTER_COL).indices.items():
#         centroid = E[df["row"].values[idx]].mean(axis=0); centroid /= np.linalg.norm(centroid)
#         cands = topics[(topics["condition"] == c) & (topics["cluster"] == cl)]["ctfidf_top30"].iloc[0].split(" | ")
#         top = keybert_rank(centroid, np.vstack([cand_emb[t] for t in cands]), cands, top_n=5, diversity=0.5)
#         rows.append({"condition": c, "cluster": cl, "n_reviews": len(idx), "sbert_keywords": " | ".join(top)})
# auto = pd.DataFrame(rows)
# auto.to_csv(f"{OUTPUT_DIR}/cluster_labels_sbert_auto.csv", index=False)
# print(auto.to_string(index=False))
def keybert_rank(centroid, cand_vecs, cands, top_n=5, diversity=0.5, baseline=None):
    """Pick top_n candidate terms that are close to the cluster centroid, different from each other (MMR),
    and specific to the cluster: if `baseline` (centroid of the whole condition) is given, its similarity is subtracted.
    score(i) = (1 - diversity) * [cos(cand_i, centroid) - cos(cand_i, baseline)] - diversity * max cos(cand_i, already chosen)"""
    sims = cand_vecs @ centroid
    if baseline is not None:
        sims = sims - cand_vecs @ baseline
    chosen = [int(np.argmax(sims))]
    while len(chosen) < min(top_n, len(cands)):
        rest = [i for i in range(len(cands)) if i not in chosen]
        redundancy = (cand_vecs[rest] @ cand_vecs[chosen].T).max(axis=1)
        score = (1 - diversity) * sims[rest] - diversity * redundancy
        chosen.append(rest[int(np.argmax(score))])
    return [cands[i] for i in chosen]


# embeddings de todos los candidatos (una sola vez)
all_cands = sorted({t for s_ in topics["ctfidf_top30"] for t in s_.split(" | ")})
cand_emb = dict(zip(all_cands, model.encode(all_cands, batch_size=BATCH_SIZE, normalize_embeddings=True)))

rows = []
for c in CONDITIONS:
    E   = np.load(f"{OUTPUT_DIR}/{EMB}_{key(c)}.npy")
    ids = pd.read_parquet(f"{OUTPUT_DIR}/{EMB}_{key(c)}_ids.parquet").reset_index(drop=True)
    ids["row"] = range(len(ids))
    df = (ids.merge(clusters[[ID, CLUSTER_COL]], on=ID)
             .sort_values("row").reset_index(drop=True))
    cond_centroid = E.mean(axis=0); cond_centroid /= np.linalg.norm(cond_centroid)      # centro de toda la condición
    for cl, idx in df.groupby(CLUSTER_COL).indices.items():
        centroid = E[df["row"].values[idx]].mean(axis=0); centroid /= np.linalg.norm(centroid)
        cands = topics[(topics["condition"] == c) & (topics["cluster"] == cl)]["ctfidf_top30"].iloc[0].split(" | ")
        top = keybert_rank(centroid, np.vstack([cand_emb[t] for t in cands]), cands,
                           top_n=5, diversity=0.5, baseline=cond_centroid)
        rows.append({"condition": c, "cluster": cl, "n_reviews": len(idx), "sbert_keywords": " | ".join(top)})
auto = pd.DataFrame(rows)
auto.to_csv(f"{OUTPUT_DIR}/cluster_labels_sbert_auto.csv", index=False)
print(auto.to_string(index=False))

    condition  cluster  n_reviews                                               sbert_keywords
   depression        0        230                 taking cymbalta | years | doctor | pain | im
   depression        1        249                        pristiq | better | ive | felt | years
   depression        2        208        effexor xr | years | withdrawal symptoms | ive | felt
   depression        3        272                      wellbutrin | years | xl | weight | good
   depression        4        184               mood swings | improved | felt | doctor | month
   depression        5        227                 suicidal thoughts | time | going | dont | im
   depression        6        365                      asleep | time | medication | ive | fall
   depression        7         88                  high blood | day | heart | weight | anxiety
   depression        8        163                    viibryd | diarrhea | sexual | far | tried
   depression        9        151                s

## C6 · Table of B's clusters

A summary of each of B's 47 k-means clusters. This is a helper step: it does not use the embeddings and it is not an evaluation. It builds one row per cluster with:

- `condition` and `cluster`: which cluster it is (for example, Birth Control, cluster 3).
- `n_reviews`: how many reviews it contains.
- `mean_rating`: the average rating of those reviews.
- `top_drugs`: the three drugs that appear most often in the cluster.

For each condition, the cell loads the review IDs, joins the drug and rating (from A) and the cluster of each review (from B), and groups by cluster.

**What it is used for:**
1. It is the base of the final table for D (C10.2, `cluster_labels_final.csv`), to which the keywords, the NPMI and the cohesion are added later, so that D receives everything in one file.
2. It shows at a glance whether a cluster is dominated by one drug (for example, `top_drugs` with a single name repeated), which supports the finding that some clusters group by drug rather than by experience.

In [9]:
rows = []
for c in CONDITIONS:
    ids = pd.read_parquet(f"{OUTPUT_DIR}/{EMB}_{key(c)}_ids.parquet")
    df = (ids.merge(reviews[[ID, "drug_name", "rating"]], on=ID).merge(clusters[[ID, CLUSTER_COL]], on=ID))
    for cl, sub in df.groupby(CLUSTER_COL):
        rows.append({"condition": c, "cluster": cl, "n_reviews": len(sub), "mean_rating": round(sub["rating"].mean(), 2),
                     "top_drugs": ", ".join(sub["drug_name"].value_counts().head(3).index)})
to_label = pd.DataFrame(rows)
print(to_label.shape)   # (47, 5)
print(to_label.to_string(index=False))

(100, 5)
    condition  cluster  n_reviews  mean_rating                                                                                               top_drugs
   depression        0        230         7.27                                                                       Duloxetine, Vortioxetine, Abilify
   depression        1        249         7.51                                                                    Desvenlafaxine, Viibryd, Venlafaxine
   depression        2        208         7.27                                                                           Venlafaxine, Effexor, Viibryd
   depression        3        272         7.51                                                                    Wellbutrin, Wellbutrin XL, Bupropion
   depression        4        184         6.99                                                                  Citalopram, Desvenlafaxine, Sertraline
   depression        5        227         6.02                                       

## C7 · Automatic metrics for each cluster

We now have 47 clusters and keywords for each of them, but we still need to know **whether the clusters are good**. Without manual labelling, we measure two things with numbers.

### 1. NPMI coherence: do the words of the cluster go together?

We take the 10 most distinctive terms of the cluster (from C4) and, **for every pair of terms**, we check whether they appear **in the same reviews** more often than chance would explain.

- If "weight" and "gain" appear together in many reviews, their NPMI is high.
- If two terms are unrelated and only meet by chance, their NPMI is close to 0.
- If they almost never appear together, it is negative.

NPMI goes from −1 to 1, and it is the standard measure of topic coherence. The NPMI of a cluster is the average over all its pairs of terms.

**How do we know what counts as "high"?** A value of 0.15 means nothing on its own, so we compute a **random baseline**: 200 sets of 10 terms chosen at random among the candidate terms of the same condition, and the average NPMI of those sets. If the clusters score clearly above this baseline, their terms really go together.

### 2. Cohesion in meaning space: is the cluster well separated?

This metric uses the embeddings instead of the words.

- For every review we measure how similar it is to the **centre of its own cluster** (cosine similarity) and how similar it is to the **centre of the closest other cluster** of the same condition.
- The cohesion gap is the difference between the two, averaged over the reviews of the cluster.

A large positive value means that the reviews are clearly closer to their own centre than to any other, so the cluster is well separated from its neighbours. A value close to 0 means the cluster overlaps with another one. If the values are high, the meaning space (BioLORD) **agrees** with the clusters that B built from words.

### How to read the results

Only the NPMI is compared with a random baseline; the cohesion gap is read through its sign and size. If the NPMI is clearly above the baseline and the cohesion gap is clearly positive, the clusters are coherent in a statistical sense. This is an automatic validation: there is no manual labelling.

**Limitations:**
- The terms used for the NPMI were chosen because they are frequent in the cluster, which favours a high score from the start. The random baseline compensates for this only partly.
- Neither metric tells us whether a keyword label makes sense to a person; they only show whether the cluster is statistically coherent.

In [10]:
def npmi_of(X, eps=1e-12):
    # X: matriz binaria dispersa (reseñas × términos). NPMI medio sobre todos los pares de términos.
    X = X.tocsc(); n = X.shape[0]; dfc = np.asarray(X.sum(axis=0)).ravel(); vals = []
    for i in range(X.shape[1]):
        for j in range(i + 1, X.shape[1]):
            pa, pb = dfc[i] / n, dfc[j] / n
            pab = float(X[:, i].multiply(X[:, j]).sum()) / n
            if pab <= 0 or pa == 0 or pb == 0: vals.append(-1.0)
            elif pab >= 1: vals.append(1.0)
            else: vals.append(np.log(pab / (pa * pb)) / (-np.log(pab)))
    return float(np.mean(vals)) if vals else float("nan")

rng = np.random.default_rng(0); met_rows = []
for c in CONDITIONS:
    E_  = np.load(f"{OUTPUT_DIR}/{EMB}_{key(c)}.npy")
    ids_ = pd.read_parquet(f"{OUTPUT_DIR}/{EMB}_{key(c)}_ids.parquet").reset_index(drop=True); ids_["row"] = range(len(ids_))
    d_ = (ids_.merge(reviews[[ID, TEXT]], on=ID).merge(clusters[[ID, CLUSTER_COL]], on=ID).sort_values("row").reset_index(drop=True))
    tc = topics[topics.condition == c]
    pool = sorted({t for s_ in tc["ctfidf_top30"] for t in s_.split(" | ")})
    cv_ = CountVectorizer(vocabulary=pool, stop_words="english", ngram_range=(1, 2), binary=True)
    Xall = cv_.fit_transform(d_[TEXT].astype(str)).tocsc(); col = {t: i for i, t in enumerate(pool)}
    rand_scores = []
    for _ in range(200):
        sel = list(rng.choice(pool, 10, replace=False))
        rand_scores.append(npmi_of(Xall[:, [col[t] for t in sel]]))
    cl_ids = sorted(d_[CLUSTER_COL].unique())
    cents = np.vstack([E_[d_.loc[d_[CLUSTER_COL] == cl, "row"].values].mean(axis=0) for cl in cl_ids])
    cents /= np.linalg.norm(cents, axis=1, keepdims=True)
    for k_, cl in enumerate(cl_ids):
        terms = [t for t in tc[tc.cluster == cl]["ctfidf_top10"].iloc[0].split(" | ") if t in col]
        coh = npmi_of(Xall[:, [col[t] for t in terms]])
        mine = E_[d_.loc[d_[CLUSTER_COL] == cl, "row"].values]
        own = float((mine @ cents[k_]).mean())
        others = np.delete(np.arange(len(cl_ids)), k_)
        nearest = float((mine @ cents[others].T).max(axis=1).mean()) if len(others) else float("nan")
        met_rows.append({"condition": c, "cluster": cl, "npmi": round(coh, 3), "npmi_random_mean": round(float(np.mean(rand_scores)), 3),
                         "cohesion_own": round(own, 3), "cohesion_gap": round(own - nearest, 3)})
metrics = pd.DataFrame(met_rows)
metrics.to_csv(f"{OUTPUT_DIR}/cluster_coherence_metrics.csv", index=False)
print(metrics.groupby("condition")[["npmi", "npmi_random_mean", "cohesion_gap"]].mean().round(3))
run_stats["npmi_mean_clusters"] = round(float(metrics["npmi"].mean()), 3)
run_stats["npmi_mean_random_baseline"] = round(float(metrics["npmi_random_mean"].mean()), 3)
run_stats["cohesion_gap_mean"] = round(float(metrics["cohesion_gap"].mean()), 3)
print("\nNPMI medio de los clusters:", run_stats["npmi_mean_clusters"], "| línea base aleatoria:", run_stats["npmi_mean_random_baseline"])

                npmi  npmi_random_mean  cohesion_gap
condition                                           
acne           0.139            -0.131        -0.018
birth control  0.146            -0.079         0.009
depression     0.113            -0.079        -0.007
pain           0.151            -0.360        -0.007

NPMI medio de los clusters: 0.137 | línea base aleatoria: -0.165


## C8 · Why k-means and not DBSCAN

B ran two clustering algorithms on the same reviews: **k-means** assigns every review to one of k groups (11–12 per condition, 47 clusters in total), while **DBSCAN** looks for dense regions and labels everything else as **noise** (`-1`). This section explains why I describe only the k-means clusters, and the cell below prints the share of noise of DBSCAN for each condition.

B's DBSCAN results do not suit this purpose:
- In **Birth Control**, about 99% of the reviews are noise, so there is nothing to describe.
- In the other conditions, DBSCAN finds only a few clusters (2 to 7, compared with 11–12 for k-means) and leaves 5% to 20% of the reviews out, which means leaving those patients' experiences out of the analysis.
- B also prefers k-means because it gives **full coverage** and because the Davies-Bouldin index tends to favour convex clusters over density-based ones.

Therefore I work with the 47 k-means clusters.

In [11]:
print("% ruido DBSCAN (-1) por condición:")
print(clusters.groupby(COND)["dbscan_cluster"].apply(lambda s: (s == -1).mean().round(3)))
print("\nnº de valores distintos de dbscan_cluster:")
print(clusters.groupby(COND)["dbscan_cluster"].nunique())
for c, g in clusters.groupby(COND):
    print(c, g["dbscan_cluster"].value_counts().head(5).to_dict())

% ruido DBSCAN (-1) por condición:
condition_name
Acne             0.261
Birth Control    0.114
Depression       0.193
Pain             0.198
Name: dbscan_cluster, dtype: float64

nº de valores distintos de dbscan_cluster:
condition_name
Acne             4
Birth Control    9
Depression       3
Pain             3
Name: dbscan_cluster, dtype: int64
Acne {0: 3354, -1: 1187, 1: 4, 2: 4}
Birth Control {0: 17282, -1: 2217, 5: 7, 3: 4, 2: 3}
Depression {0: 5732, -1: 1375, 1: 2}
Pain {0: 4152, -1: 1029, 1: 6}


## C9 · Clustering in meaning space: does Sentence-BERT group by experience or by drug?

**Question:** if I re-cluster the reviews with Sentence-BERT embeddings (using the same k as B for each condition), do the clusters differ from B's word-based clusters, and do they describe experiences or just drugs?

**Metric: drug purity.** For each cluster, the share of its reviews that belong to its most frequent `drug_name`, averaged over the clusters. A high value means the cluster is basically "one drug". The baseline is the share of the most frequent drug in the whole condition: if the purity only matches this baseline, the clustering does not depend on the drug.

**Other metrics:**
- **ARI** (adjusted Rand index): how much my clusters agree with B's (1 = identical, 0 = as expected by chance).
- **Davies-Bouldin index:** how well separated the clusters are (lower is better). It is computed for B's clusters *inside the embedding space* (`DB_B_in_sbert_space`) and for the Sentence-BERT k-means clusters (`DB_sbert`), so both are compared with the same yardstick. Note that B computed its own DB in the TF-IDF/SVD space, where the numbers are not comparable with the ones here.

**Two versions:**
1. **Unmasked Sentence-BERT.** Compare `purity_sbert_unmasked` with `baseline_top_drug_share` to see how much the grouping depends on the drug.
2. **Sentence-BERT with drug names replaced by `medication`.** The list of names is built **only from project data** (`drug_name`, `drug_name_aliases`, `brand_generic_drug_name_pairs.csv`, `drug_families.csv`); nothing is written from memory.

**Limits:** about 4.8% of the reviews still contain a drug name that is not masked (brands that are not present in those files); there is no "ground truth" to say which clustering is better; and the numbers depend on the masking list. I present this as a **finding and a limitation**, not as a demonstrated improvement.

In [12]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score

def drug_purity(df, col):
    return df.groupby(col)["drug_name"].agg(lambda s: s.value_counts(normalize=True).iloc[0]).mean()

# --- versión 1: SBERT sin enmascarar ---
res_unmasked, assign_unmasked = [], []
for c in CONDITIONS:
    E   = np.load(f"{OUTPUT_DIR}/{EMB}_{key(c)}.npy")
    ids = pd.read_parquet(f"{OUTPUT_DIR}/{EMB}_{key(c)}_ids.parquet").reset_index(drop=True); ids["row"] = range(len(ids))
    df = (reviews[reviews["cond_l"] == c].merge(ids, on=ID).merge(clusters[[ID, CLUSTER_COL]], on=ID)
          .sort_values("row").reset_index(drop=True))
    E = E[df["row"].values]
    k = df[CLUSTER_COL].nunique()
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(E)
    df["sbert_cluster"] = km.labels_
    res_unmasked.append({"condition": c, "k": k,
        "baseline_top_drug_share": round(df["drug_name"].value_counts(normalize=True).iloc[0], 3),
        "purity_B": round(drug_purity(df, CLUSTER_COL), 3),
        "purity_sbert_unmasked": round(drug_purity(df, "sbert_cluster"), 3),
        "DB_sbert": round(davies_bouldin_score(E, km.labels_), 3),
        "DB_B_in_sbert_space": round(davies_bouldin_score(E, df[CLUSTER_COL]), 3),
        "ARI_vs_B": round(adjusted_rand_score(df[CLUSTER_COL], df["sbert_cluster"]), 3),
        "rating_spread_B": round(df.groupby(CLUSTER_COL)["rating"].mean().std(), 3),
        "rating_spread_sbert": round(df.groupby("sbert_cluster")["rating"].mean().std(), 3)})
    assign_unmasked.append(df[[ID, COND, "sbert_cluster"]])
res_unmasked = pd.DataFrame(res_unmasked)
res_unmasked.to_csv(f"{OUTPUT_DIR}/sbert_vs_tfidf_summary.csv", index=False)
pd.concat(assign_unmasked).to_parquet(f"{OUTPUT_DIR}/sbert_cluster_assignments.parquet")
print(res_unmasked.to_string(index=False))

    condition  k  baseline_top_drug_share  purity_B  purity_sbert_unmasked  DB_sbert  DB_B_in_sbert_space  ARI_vs_B  rating_spread_B  rating_spread_sbert
   depression 26                    0.093     0.267                  0.484     3.056                7.856     0.101            1.027                1.052
birth control 23                    0.148     0.423                  0.439     3.346                6.122     0.163            1.158                1.395
         pain 26                    0.085     0.260                  0.293     3.435                8.577     0.068            1.009                0.862
         acne 25                    0.121     0.321                  0.429     3.312                8.302     0.082            1.079                1.264


In [13]:
# --- lista de nombres de fármaco SOLO a partir de los datos del proyecto ---
from collections import Counter
pairs = pd.read_csv(find_input("brand_generic_drug_name_pairs.csv"))
fam   = pd.read_csv(find_input("drug_families.csv"))

def split_names(v):
    for part in re.split(r"\s*\|\s*|\s*/\s*", str(v)):      # los archivos separan con " | " y " / "
        p = part.strip().lower()
        if len(p) >= 4:
            yield p

names = set()
for _, r in reviews.drop_duplicates("drug_name")[["drug_name", "drug_name_aliases"]].iterrows():
    for n in [r["drug_name"]] + list(r["drug_name_aliases"]):
        names.update(split_names(n))
for t in (pairs, fam):
    for v in t["drug_names"].dropna():
        names.update(split_names(v))

EXCLUDE = {"commit"}     # palabras corrientes ("commit suicide"); añade aquí otras que veas en la lista de abajo
names -= EXCLUDE
pattern = re.compile(r"\b(?:" + "|".join(re.escape(n) for n in sorted(names, key=len, reverse=True)) + r")\b",
                     flags=re.IGNORECASE)
mask = lambda t: pattern.sub("medication", t)

cuatro = reviews[reviews["cond_l"].isin(CONDITIONS)]
cnt = Counter(m.lower() for t in cuatro[TEXT] for m in pattern.findall(t))
print(len(names), "nombres en la lista;", len(cnt), "aparecen en las 4 condiciones")
print("Los más tapados (revisa que ninguno sea una palabra corriente):")
print(cnt.most_common(60))

3460 nombres en la lista; 708 aparecen en las 4 condiciones
Los más tapados (revisa que ninguno sea una palabra corriente):
[('nexplanon', 2195), ('mirena', 2074), ('implanon', 1720), ('skyla', 1321), ('nuvaring', 1081), ('zoloft', 958), ('lexapro', 915), ('accutane', 914), ('wellbutrin', 787), ('prozac', 769), ('pristiq', 732), ('cymbalta', 726), ('sprintec', 723), ('epiduo', 705), ('ibuprofen', 681), ('effexor', 634), ('tramadol', 625), ('celexa', 578), ('paragard', 566), ('percocet', 461), ('paxil', 400), ('yasmin', 385), ('lutera', 381), ('vicodin', 377), ('doxycycline', 366), ('viibryd', 366), ('norco', 364), ('aviane', 352), ('oxycodone', 352), ('oxycontin', 337), ('trinessa', 334), ('apri', 303), ('morphine', 297), ('tylenol', 293), ('dilaudid', 284), ('citalopram', 259), ('methadone', 258), ('nucynta', 250), ('xanax', 249), ('aczone', 243), ('minocycline', 242), ('copper', 238), ('abilify', 238), ('sertraline', 236), ('tri-sprintec', 234), ('advil', 231), ('beyaz', 224), ('hydr

In [14]:
# --- versión 2: SBERT con nombres enmascarados (lento la primera vez con BioLORD; luego se reutiliza el .npy) ---
res_masked, assign_masked = [], []
for c in CONDITIONS:
    df = (reviews[reviews["cond_l"] == c].merge(clusters[[ID, CLUSTER_COL]], on=ID).reset_index(drop=True))
    f = f"{OUTPUT_DIR}/{EMB_MASKED}_{key(c)}.npy"
    if os.path.exists(f):
        E = np.load(f)
    else:
        E = model.encode(df[TEXT].map(mask).tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
        np.save(f, E)
    df[[ID]].to_parquet(f"{OUTPUT_DIR}/{EMB_MASKED}_{key(c)}_ids.parquet")
    k = df[CLUSTER_COL].nunique()
    df["sbert_masked_cluster"] = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(E)
    res_masked.append({"condition": c, "k": k,
        "baseline_top_drug_share": round(df["drug_name"].value_counts(normalize=True).iloc[0], 3),
        "purity_B": round(drug_purity(df, CLUSTER_COL), 3),
        "purity_sbert_masked_v2": round(drug_purity(df, "sbert_masked_cluster"), 3),
        "ARI_vs_B": round(adjusted_rand_score(df[CLUSTER_COL], df["sbert_masked_cluster"]), 3),
        "rating_spread_B": round(df.groupby(CLUSTER_COL)["rating"].mean().std(), 3),
        "rating_spread_masked": round(df.groupby("sbert_masked_cluster")["rating"].mean().std(), 3)})
    assign_masked.append(df[[ID, COND, "sbert_masked_cluster"]])
res_masked = pd.DataFrame(res_masked)
res_masked.to_csv(f"{OUTPUT_DIR}/sbert_masked_v2_summary.csv", index=False)
pd.concat(assign_masked).to_parquet(f"{OUTPUT_DIR}/sbert_masked_v2_cluster_assignments.parquet")
print(res_masked.to_string(index=False))

# fuga residual: % de reseñas que contienen algún nombre de fármaco de esos archivos tras enmascarar
leak = cuatro[TEXT].map(mask).str.lower().map(lambda t: bool(pattern.search(t))).mean()
print(f"\nReseñas con nombre de fármaco aún visible tras enmascarar: {leak:.1%}")

    condition  k  baseline_top_drug_share  purity_B  purity_sbert_masked_v2  ARI_vs_B  rating_spread_B  rating_spread_masked
   depression 26                    0.093     0.267                   0.143     0.026            1.027                 1.635
birth control 23                    0.148     0.423                   0.255     0.087            1.158                 1.691
         pain 26                    0.085     0.260                   0.151     0.041            1.009                 0.866
         acne 25                    0.121     0.321                   0.311     0.036            1.079                 1.352

Reseñas con nombre de fármaco aún visible tras enmascarar: 0.0%


**How to read C9 (for the report):** compare `purity_sbert_unmasked` with `baseline_top_drug_share` (the baseline): if it is well above the baseline, the model groups by drug. Then compare `purity_sbert_masked_v2`: if it drops, masking the names reduces the dependence on the drug name. A low ARI against B means that the two clusterings are different. This does **not** show that one is better; it only shows that Sentence-BERT and TF-IDF find different structures. Also, B's clusters partly group by drug as well (`purity_B` > baseline), so this is not a weakness of Sentence-BERT alone. Numbers from an earlier version with MiniLM no longer apply: quote only the numbers from this run.

## C10 · Deliverables

### C10.1 Embeddings for A (similarity search by meaning)

A single file with **one vector per `review_id`**: the 4 conditions and, if `EMBED_ALL_OTHER_REVIEWS=True`, also all the other reviews (A's similar-story search uses all 128,663). There are two versions: unmasked (`embeddings_all`) and masked v2 (`embeddings_masked_v2_all`).

**To agree with A:**
- The **unmasked** version covers every review, but its vectors contain the drug name, so LSH will tend to pair reviews of the same drug.
- The **masked v2** version avoids this, but it only exists for the 4 conditions.
- A already removes drug names in his word-based version, so the consistent option would be to mask the other reviews as well. That means recomputing the embeddings of the other reviews with masking (`mask`).

In [15]:
def concat_embeddings(prefix, out_name, extra=()):
    E_all, id_all = [], []
    for c in CONDITIONS:
        E_all.append(np.load(f"{OUTPUT_DIR}/{prefix}_{key(c)}.npy"))
        id_all.append(pd.read_parquet(f"{OUTPUT_DIR}/{prefix}_{key(c)}_ids.parquet")[[ID]])
    for name in extra:                                   # reseñas de otras condiciones (si existen)
        if os.path.exists(f"{OUTPUT_DIR}/{name}.npy"):
            E_all.append(np.load(f"{OUTPUT_DIR}/{name}.npy"))
            id_all.append(pd.read_parquet(f"{OUTPUT_DIR}/{name}_ids.parquet")[[ID]])
    E_all = np.vstack(E_all).astype("float32"); ids_all = pd.concat(id_all, ignore_index=True)
    assert len(E_all) == len(ids_all) and ids_all[ID].is_unique
    np.save(f"{OUTPUT_DIR}/{out_name}.npy", E_all)
    ids_all.to_parquet(f"{OUTPUT_DIR}/{out_name}_ids.parquet")
    return E_all.shape

shape_unmasked = concat_embeddings(EMB, "embeddings_all", extra=[EMB_OTHER])
shape_masked   = concat_embeddings(EMB_MASKED, "embeddings_masked_v2_all")   # solo las 4 condiciones
print("sin enmascarar:", shape_unmasked, "| enmascarado v2:", shape_masked)
if shape_unmasked[0] < len(reviews):
    print(f"AVISO: embeddings_all cubre {shape_unmasked[0]:,} de {len(reviews):,} reseñas: avisa a A.")

sin enmascarar: (36365, 768) | enmascarado v2: (36365, 768)
AVISO: embeddings_all cubre 36,365 de 128,663 reseñas: avisa a A.


### C10.2 Final labels for D

A CSV with **one row per cluster of B** (`cluster_labels_final.csv`), with these columns:
- `condition` (lower case), `cluster`, `n_reviews`, `mean_rating` and `top_drugs` (from C6).
- `sbert_keywords`: the five Sentence-BERT keywords of the cluster (C5).
- `ctfidf_top5`: the five best c-TF-IDF terms (C4).
- `npmi` and `cohesion_gap`: the automatic quality metrics of the cluster (C7).
- `label_final`: the first three Sentence-BERT keywords, joined with " / ". It is a keyword label chosen automatically, not a sentence written by a person (`label_source` says so).
- `b_top_terms`: B's top TF-IDF terms for the same cluster, so that both views can be compared.

D only has to join this file to `cluster_assignments.parquet` on `(condition, cluster)`: the condition in lower case against `condition_name` in lower case, and `cluster` against `kmeans_cluster`.

In [16]:
base = to_label[["condition", "cluster", "n_reviews", "mean_rating", "top_drugs"]].merge(
    auto[["condition", "cluster", "sbert_keywords"]], on=["condition", "cluster"]).merge(
    topics[["condition", "cluster", "ctfidf_top5"]], on=["condition", "cluster"], how="left").merge(
    metrics[["condition", "cluster", "npmi", "cohesion_gap"]], on=["condition", "cluster"], how="left")
base["label_final"] = base["sbert_keywords"].str.split(" | ", regex=False).str[:3].str.join(" / ")
base["label_source"] = "auto_keywords_sbert"
prof = pd.read_csv(find_input("cluster_profiles.csv"))          # términos TF-IDF de B (B7)
prof["condition"] = prof["condition"].str.lower().str.strip()
base = base.merge(prof[["condition", "cluster", "top_5_terms"]].rename(columns={"top_5_terms": "b_top_terms"}),
                  on=["condition", "cluster"], how="left")
base.to_csv(f"{OUTPUT_DIR}/cluster_labels_final.csv", index=False)
print(base[["condition", "cluster", "n_reviews", "label_final", "npmi", "cohesion_gap"]].to_string(index=False))

    condition  cluster  n_reviews                              label_final   npmi  cohesion_gap
   depression        0        230         taking cymbalta / years / doctor  0.081         0.112
   depression        1        249                   pristiq / better / ive  0.071         0.138
   depression        2        208 effexor xr / years / withdrawal symptoms  0.156         0.090
   depression        3        272                  wellbutrin / years / xl  0.124         0.084
   depression        4        184            mood swings / improved / felt  0.092        -0.049
   depression        5        227         suicidal thoughts / time / going  0.097        -0.037
   depression        6        365               asleep / time / medication  0.113        -0.031
   depression        7         88                 high blood / day / heart  0.205        -0.035
   depression        8        163              viibryd / diarrhea / sexual  0.123         0.084
   depression        9        151       

### C10.3 Run statistics

I save every number that I will quote in the report into one JSON file, `role_c_run_stats.json`, in the same way as A and B. It includes the model and the embedding size, the number of reviews embedded, the results of C4 (overlap with B's terms and the share of keywords that are drug or condition names), C7 (NPMI of the clusters against the random baseline, and the cohesion gap), C9 (drug purity, ARI and Davies-Bouldin, with and without drug names), the share of drug names left unmasked, the DBSCAN noise share, the model comparison and the run time.

In [17]:
run_stats.update({
  "model": MODEL_NAME, "embedding_dim": int(shape_unmasked[1]),
  "n_reviews_embedded": int(shape_unmasked[0]), "conditions": CONDITIONS,
  "n_clusters": int(len(to_label)),
  "sbert_unmasked": res_unmasked.to_dict(orient="records"),
  "sbert_masked_v2": res_masked.to_dict(orient="records"),
  "masked_names_in_list": int(len(names)),
  "residual_unmasked_name_share": round(float(leak), 3),
  "dbscan_noise_share": clusters.groupby(COND)["dbscan_cluster"].apply(lambda s: round(float((s == -1).mean()), 3)).to_dict(),
  "model_comparison_sample": model_comparison.to_dict(orient="records") if "model_comparison" in globals() else None,
  "runtime_minutes": round((time.perf_counter() - NOTEBOOK_STARTED) / 60, 1)})
stats = run_stats
(OUTPUT_DIR / "role_c_run_stats.json").write_text(json.dumps(stats, indent=2, default=str))
print(json.dumps(stats, indent=1, default=str)[:3000])

{
 "n_reviews_all": 128663,
 "n_reviews_in_scope": 36365,
 "share_top10_terms_that_are_drug_or_condition_names": 0.143,
 "clusters_with_name_in_top5": 51,
 "ctfidf_word_overlap_with_B_mean": 0.677,
 "npmi_mean_clusters": 0.137,
 "npmi_mean_random_baseline": -0.165,
 "cohesion_gap_mean": -0.006,
 "model": "FremyCompany/BioLORD-2023",
 "embedding_dim": 768,
 "n_reviews_embedded": 36365,
 "conditions": [
  "depression",
  "birth control",
  "pain",
  "acne"
 ],
 "n_clusters": 100,
 "sbert_unmasked": [
  {
   "condition": "depression",
   "k": 26,
   "baseline_top_drug_share": 0.093,
   "purity_B": 0.267,
   "purity_sbert_unmasked": 0.484,
   "DB_sbert": 3.056,
   "DB_B_in_sbert_space": 7.856,
   "ARI_vs_B": 0.101,
   "rating_spread_B": 1.027,
   "rating_spread_sbert": 1.052
  },
  {
   "condition": "birth control",
   "k": 23,
   "baseline_top_drug_share": 0.148,
   "purity_B": 0.423,
   "purity_sbert_unmasked": 0.439,
   "DB_sbert": 3.346,
   "DB_B_in_sbert_space": 6.122,
   "ARI_vs_B": 

## Where the results are

| File | Contents | Used by |
|---|---|---|
| `outputs/emb_biolord_{condition}.npy` + `_ids.parquet` | embeddings per condition (unmasked) | C4–C9 |
| `outputs/embeddings_all.npy` + `embeddings_all_ids.parquet` | **one vector (768 numbers) per `review_id`** (4 conditions + all other reviews if `EMBED_ALL_OTHER_REVIEWS`) | **A** |
| `outputs/embeddings_masked_v2_all.npy` + `_ids.parquet` | same, with drug names masked (4 conditions only) | A (optional) |
| `outputs/cluster_topics_ctfidf.csv` | c-TF-IDF terms per cluster and overlap with B's terms | report |
| `outputs/cluster_labels_sbert_auto.csv` | 5 keywords per cluster (Sentence-BERT, KeyBERT-style) | report |
| `outputs/cluster_coherence_metrics.csv` | NPMI, random baseline and cohesion per cluster | C7, report |
| **`outputs/cluster_labels_final.csv`** | **final label per cluster of B** | **D** |
| `outputs/sbert_vs_tfidf_summary.csv`, `sbert_masked_v2_summary.csv`, `model_comparison_sample.csv` | evaluation of the clustering in meaning space | report |
| `outputs/role_c_run_stats.json` | all the numbers | report |

## How to use my outputs in your own notebook

```python
from pathlib import Path
import numpy as np
import pandas as pd

folder = next(p for p in [Path('/kaggle/input'), Path('../nlp-embeddings/outputs')] if p.exists())

# --- A: embeddings por review_id ---
E   = np.load(next(folder.rglob('embeddings_all.npy')))                    # (n_reviews, 768), norma 1
ids = pd.read_parquet(next(folder.rglob('embeddings_all_ids.parquet')))    # misma fila = mismo review_id
vec = dict(zip(ids['review_id'], E))                                        # vec[review_id] -> 768 números
# similitud coseno entre dos reseñas = vec[a] @ vec[b]

# --- D: etiqueta legible de cada cluster ---
lab = pd.read_csv(next(folder.rglob('cluster_labels_final.csv')))
# columnas: condition (minúsculas), cluster, n_reviews, mean_rating, top_drugs, sbert_keywords, ctfidf_top5, npmi, cohesion_gap, label_final, label_source, b_top_terms
# se une a cluster_assignments.parquet por (condition_name en minúsculas, kmeans_cluster)
```

## Limitations (for the report)
- One main model (BioLORD-2023). Very long reviews are truncated to the maximum length of the model.
- The c-TF-IDF terms depend on the vocabulary of each condition; clusters organised by drug produce terms that are drug names.
- The keywords of C5 can only be terms that already appear in the c-TF-IDF of C4; they are a suggestion, not a final label.
- There is no manual labelling: the evaluation relies on automatic metrics (NPMI against a random baseline, cohesion, DB, purity, ARI) and on a qualitative pilot review of some clusters.
- The keywords sometimes include generic words and drug or condition names (measured in C4 with `share_name_terms_top10`); they are not filtered by hand because any list would be arbitrary.
- Some clusters mix good and bad experiences of the same topic (e.g. panic attacks, sleep): grouping by topic does not separate sentiment.
- `label_final` is made of the first 3 Sentence-BERT keywords, not a sentence written by a person; `npmi` and `cohesion_gap` indicate how far they can be trusted.
- With BioLORD, cosine similarities are generally higher than with MiniLM: a cosine threshold (e.g. for A's LSH) must be chosen by looking at the distribution, not by copying a typical value.
- Drug-name masking: the list is built only from project data; about 5% of the reviews still keep some name.
- The 4 conditions were chosen by B; Anxiety was left out (B justifies this in his section B1).

## For the report (what should come out of this notebook)
1. **New technique, clearly identified and with a brief explanation of how it works:** Sentence-BERT (why BioLORD: C9b) and c-TF-IDF (formula, implemented by hand).
2. **How it fits with the course topics of the group:** the embeddings feed A's cosine LSH, and the labels and the c-TF-IDF are applied to B's clusters (C9 evaluates them with Davies-Bouldin).
3. **Evaluation:** NPMI against a random baseline and cohesion (C7); drug purity / ARI / DB (C9); overlap of c-TF-IDF with B's terms (C4); the finding of generic words and drug/condition names among the keywords, quantified in C4.
4. **Reflection:** whether and why Sentence-BERT groups by drug, and what changes when the names are masked.
5. **Contribution table:** what each person did (completed as a group).